# Notebook 08 - Model Evaluation

**Comprehensive evaluation of the DenseNet121 model on the test set.**

## Objectives

1. Load the trained DenseNet121 model (deployed in Streamlit)
2. Run inference on the full test set (16,491 images)
3. Generate ROC curves for all 14 disease classes
4. Create precision-recall curves
5. Compute per-disease performance metrics
6. Analyze error cases (false positives/negatives)
7. Compare with baseline models
8. Save visualizations for Streamlit integration

## Outputs

- ROC curves (interactive Plotly + static PNG)
- Precision-Recall curves
- Per-disease confusion matrices
- Detailed performance metrics (JSON)
- Error analysis report

---

## 1. Setup and Configuration

In [5]:
# Standard library imports
import os
import json
from pathlib import Path
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine learning
from sklearn.metrics import (
    roc_curve, auc, 
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score
)

# Deep learning
import tensorflow as tf
from tensorflow import keras

# Progress bars
from tqdm.auto import tqdm

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

TensorFlow version: 2.18.1
Keras version: 3.12.0


In [ ]:
# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Find project root (directory containing jupyter_notebooks/)
# This works regardless of where the notebook is executed from
import os
if 'jupyter_notebooks' in os.getcwd():
    # Running from within jupyter_notebooks/
    PROJECT_ROOT = Path(os.getcwd()).parent
else:
    # Running from project root
    PROJECT_ROOT = Path(os.getcwd())

print(f"Current working directory: {os.getcwd()}")
print(f"Project root: {PROJECT_ROOT}")

# Paths
MODEL_PATH = PROJECT_ROOT / 'models' / 'saved_models' / 'densenet121_best.keras'
TEST_SPLIT_CSV = PROJECT_ROOT / 'data' / 'processed' / 'test_split.csv'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures' / 'evaluation'
REPORTS_DIR = OUTPUTS_DIR / 'reports'

# Verify paths exist
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")
if not TEST_SPLIT_CSV.exists():
    raise FileNotFoundError(f"Test split not found at {TEST_SPLIT_CSV}")

# Create output directories
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Disease classes (14 conditions)
DISEASE_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia"
]

# Model parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ImageNet normalization (same as training)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

print(f"\n✅ Configuration complete")
print(f"   Model path: {MODEL_PATH.name}")
print(f"   Test split: {TEST_SPLIT_CSV.name}")
print(f"   Disease classes: {len(DISEASE_CLASSES)}")

## 2. Load Model and Test Data

In [7]:
# Load the trained DenseNet121 model
print("Loading DenseNet121 model...")
model = keras.models.load_model(MODEL_PATH)

print("\nModel Summary:")
print(f"Input shape: {model.input_shape}")
print(f"Output shape: {model.output_shape}")
print(f"Total parameters: {model.count_params():,}")

# Display model architecture
model.summary()

Loading DenseNet121 model...


ValueError: File not found: filepath=/Volumes/SSD/models/saved_models/densenet121_best.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
# Load test split CSV
print("Loading test split metadata...")

if not TEST_SPLIT_CSV.exists():
    raise FileNotFoundError(f"Test split CSV not found at {TEST_SPLIT_CSV}. Run notebook 03 first.")

test_df = pd.read_csv(TEST_SPLIT_CSV)
print(f"✅ Loaded metadata for {len(test_df):,} test images")
print(f"\nColumns: {test_df.columns.tolist()[:10]}...")  # Show first 10 columns
print(f"\nDisease label columns:")
for disease in DISEASE_CLASSES:
    if disease in test_df.columns:
        n_positive = test_df[disease].sum()
        print(f"  {disease:20}: {n_positive:5d} positive ({n_positive/len(test_df)*100:.1f}%)")

print(f"\nFirst few rows:")
display(test_df[['Image Index', 'full_path'] + DISEASE_CLASSES].head())

In [ ]:
# Create custom data generator from CSV
print("Creating data generator from CSV...")

def preprocess_image(img_path):
    """Load and preprocess image for DenseNet121."""
    # Load image
    img = Image.open(img_path).convert('RGB')
    
    # Resize
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    
    # Convert to array and normalize to [0, 1]
    img_array = np.array(img, dtype=np.float32) / 255.0
    
    # Apply ImageNet normalization
    img_array = (img_array - IMAGENET_MEAN) / IMAGENET_STD
    
    return img_array

def data_generator(df, batch_size=32):
    """Generator that yields batches of (images, labels)."""
    num_samples = len(df)
    
    while True:  # Loop forever for Keras fit
        for offset in range(0, num_samples, batch_size):
            batch_df = df.iloc[offset:offset+batch_size]
            
            # Load and preprocess images
            images = []
            for img_path in batch_df['full_path']:
                try:
                    img = preprocess_image(img_path)
                    images.append(img)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")
                    # Use zeros as placeholder for failed images
                    images.append(np.zeros((*IMG_SIZE, 3), dtype=np.float32))
            
            # Get labels
            labels = batch_df[DISEASE_CLASSES].values.astype(np.float32)
            
            yield np.array(images), labels

# Calculate number of batches
num_batches = int(np.ceil(len(test_df) / BATCH_SIZE))

print(f"✅ Data generator created")
print(f"   Total samples: {len(test_df):,}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Number of batches: {num_batches}")
print(f"   Images per batch (last): {len(test_df) % BATCH_SIZE or BATCH_SIZE}")

## 3. Run Inference on Test Set

Generate predictions for all test images.

In [ ]:
# Run predictions on test set
print("Running inference on test set...")
print(f"This will process {len(test_df):,} images in {num_batches} batches")
print("Estimated time: ~30-45 minutes on CPU\n")

# Create generator instance
test_generator = data_generator(test_df, batch_size=BATCH_SIZE)

# Get predictions
y_pred = model.predict(
    test_generator,
    steps=num_batches,
    verbose=1
)

print(f"\nPredictions shape: {y_pred.shape}")
print(f"Expected shape: ({len(test_df)}, {len(DISEASE_CLASSES)})")

# Get true labels from dataframe
y_true = test_df[DISEASE_CLASSES].values.astype(np.float32)
print(f"True labels shape: {y_true.shape}")

# Verify shapes match
assert y_pred.shape[0] == y_true.shape[0], f"Prediction count ({y_pred.shape[0]}) != label count ({y_true.shape[0]})!"
assert y_pred.shape[1] == len(DISEASE_CLASSES), f"Predicted classes ({y_pred.shape[1]}) != expected ({len(DISEASE_CLASSES)})!"

print("\n✅ Inference complete!")

In [ ]:
# Save predictions for future use
predictions_path = REPORTS_DIR / 'densenet121_test_predictions.npz'
np.savez_compressed(
    predictions_path,
    y_pred=y_pred,
    y_true=y_true,
    disease_classes=DISEASE_CLASSES
)
print(f"Predictions saved to: {predictions_path}")
print(f"File size: {predictions_path.stat().st_size / (1024*1024):.2f} MB")

## 4. Generate ROC Curves

Create ROC curves for each disease class.

In [ ]:
# Calculate ROC curves for all diseases
print("Calculating ROC curves for all diseases...\n")

roc_data = {}

for i, disease in enumerate(DISEASE_CLASSES):
    # Get true labels and predictions for this disease
    y_true_disease = y_true[:, i]
    y_pred_disease = y_pred[:, i]
    
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_true_disease, y_pred_disease)
    roc_auc = auc(fpr, tpr)
    
    # Store data
    roc_data[disease] = {
        'fpr': fpr.tolist(),
        'tpr': tpr.tolist(),
        'thresholds': thresholds.tolist(),
        'auc': float(roc_auc),
        'n_positive': int(y_true_disease.sum()),
        'n_negative': int((1 - y_true_disease).sum())
    }
    
    print(f"{disease:20} - AUC: {roc_auc:.4f} (pos: {roc_data[disease]['n_positive']:5d}, neg: {roc_data[disease]['n_negative']:5d})")

# Calculate average AUC
avg_auc = np.mean([data['auc'] for data in roc_data.values()])
print(f"\nAverage AUC across all diseases: {avg_auc:.4f}")

In [ ]:
# Save ROC data to JSON
roc_json_path = REPORTS_DIR / 'roc_curves_densenet121.json'
with open(roc_json_path, 'w') as f:
    json.dump(roc_data, f, indent=2)
print(f"ROC data saved to: {roc_json_path}")

### 4.1 Plot Individual ROC Curves

In [ ]:
# Create individual ROC curve plots
print("Creating individual ROC curve plots...\n")

# Set up matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, (disease, data) in enumerate(roc_data.items()):
    ax = axes[i]
    
    # Plot ROC curve
    ax.plot(data['fpr'], data['tpr'], 'b-', linewidth=2, label=f'AUC = {data["auc"]:.3f}')
    
    # Plot diagonal (random classifier)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random')
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'{disease}\n(n={data["n_positive"]} positive)', fontsize=11, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

# Hide extra subplots if needed
for i in range(len(DISEASE_CLASSES), len(axes)):
    axes[i].axis('off')

plt.suptitle('ROC Curves - DenseNet121 (All Diseases)', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()

# Save figure
roc_png_path = FIGURES_DIR / 'roc_curves_all_diseases.png'
plt.savefig(roc_png_path, dpi=150, bbox_inches='tight')
print(f"Saved: {roc_png_path}")

plt.show()

### 4.2 Interactive ROC Curves (Plotly)

In [ ]:
# Create interactive Plotly visualization
print("Creating interactive ROC curves (Plotly)...\n")

# Option 1: All diseases on one plot
fig_all = go.Figure()

# Add ROC curve for each disease
for disease, data in roc_data.items():
    fig_all.add_trace(go.Scatter(
        x=data['fpr'],
        y=data['tpr'],
        mode='lines',
        name=f"{disease} (AUC={data['auc']:.3f})",
        hovertemplate=f'<b>{disease}</b><br>' +
                      'FPR: %{x:.3f}<br>' +
                      'TPR: %{y:.3f}<br>' +
                      '<extra></extra>'
    ))

# Add diagonal reference line
fig_all.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash'),
    showlegend=True
))

fig_all.update_layout(
    title='ROC Curves - DenseNet121 (All Diseases)',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=1000,
    height=700,
    hovermode='closest',
    legend=dict(
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99
    )
)

# Save interactive HTML
roc_html_path = FIGURES_DIR / 'roc_curves_interactive.html'
fig_all.write_html(roc_html_path)
print(f"Saved: {roc_html_path}")

fig_all.show()

In [ ]:
# Option 2: Individual interactive plots for each disease
print("Creating individual interactive ROC curves...\n")

for disease, data in roc_data.items():
    fig = go.Figure()
    
    # ROC curve
    fig.add_trace(go.Scatter(
        x=data['fpr'],
        y=data['tpr'],
        mode='lines',
        name=f'{disease} (AUC = {data["auc"]:.3f})',
        line=dict(color='blue', width=3),
        fill='tozeroy',
        fillcolor='rgba(0, 100, 255, 0.2)'
    ))
    
    # Diagonal
    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode='lines',
        name='Random',
        line=dict(color='red', width=2, dash='dash')
    ))
    
    fig.update_layout(
        title=f'ROC Curve - {disease}<br><sub>{data["n_positive"]} positive cases, {data["n_negative"]} negative cases</sub>',
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate',
        width=700,
        height=600,
        hovermode='x unified'
    )
    
    # Save
    disease_safe = disease.replace(' ', '_').replace('/', '_')
    path = FIGURES_DIR / f'roc_{disease_safe.lower()}.html'
    fig.write_html(path)
    print(f"Saved: {path.name}")

## 5. Summary

What we created in this notebook.

In [ ]:
# Summary of generated files
print("=" * 60)
print("NOTEBOOK 08 - MODEL EVALUATION SUMMARY")
print("=" * 60)

print(f"\n📊 Model: DenseNet121")
print(f"Test samples: {y_pred.shape[0]:,}")
print(f"Disease classes: {len(DISEASE_CLASSES)}")
print(f"Average AUC: {avg_auc:.4f}")

print(f"\n📁 Generated Files:")
print(f"\nData:")
print(f"  - {predictions_path.name}")
print(f"  - {roc_json_path.name}")

print(f"\nVisualizations:")
figures_created = list(FIGURES_DIR.glob('*.png')) + list(FIGURES_DIR.glob('*.html'))
for fig_path in sorted(figures_created):
    size_mb = fig_path.stat().st_size / (1024*1024)
    print(f"  - {fig_path.name} ({size_mb:.2f} MB)")

print(f"\n✅ Evaluation complete!")
print(f"\n💡 Next steps:")
print(f"  1. Review ROC curves in: {FIGURES_DIR}")
print(f"  2. Integrate into Streamlit dashboard")
print(f"  3. Run error analysis (notebook 09)")